# PJM Load Forecast Hourly

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add backend to path so we can import the utils
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
# Read SQL query from file
sql_path = Path.cwd() / "pjm_load_forecast.sql"
query = sql_path.read_text()

# Pull data
df = pull_from_db(query=query)
print(f"{len(df):,} rows")
df.head()

92,328 rows


,datetime,forecast_rank,forecast_execution_datetime,forecast_execution_date,forecast_date,hour_ending,region,forecast_load_mw
0,2026-03-04,41,2026-03-03 06:47:31,2026-03-03,2026-03-04,1,MIDATL,28228.0
1,2026-03-04,2,2026-03-04 10:17:17,2026-03-04,2026-03-04,1,MIDATL,29224.0
2,2026-03-04,3,2026-03-04 09:47:17,2026-03-04,2026-03-04,1,MIDATL,29224.0
3,2026-03-04,5,2026-03-04 09:17:27,2026-03-04,2026-03-04,1,MIDATL,29224.0
4,2026-03-04,6,2026-03-04 08:47:16,2026-03-04,2026-03-04,1,MIDATL,29224.0


In [3]:
df.dtypes

datetime                       datetime64[us]
forecast_rank                           int64
forecast_execution_datetime    datetime64[us]
forecast_execution_date                object
forecast_date                          object
hour_ending                             int64
region                                    str
forecast_load_mw                      float64
dtype: object

In [4]:
print("Regions:", df["region"].unique())
print("Execution dates:", sorted(df["forecast_execution_date"].unique()))

Regions: <ArrowStringArray>
['MIDATL', 'RTO', 'SOUTH', 'WEST']
Length: 4, dtype: str
Execution dates: [datetime.date(2026, 2, 25), datetime.date(2026, 2, 26), datetime.date(2026, 2, 27), datetime.date(2026, 2, 28), datetime.date(2026, 3, 1), datetime.date(2026, 3, 2), datetime.date(2026, 3, 3), datetime.date(2026, 3, 4)]


## Forecast Load by Region

In [5]:
# Latest execution date per forecast_date
df_latest = df.loc[df.groupby(["forecast_date", "hour_ending", "region"])["forecast_execution_datetime"].idxmax()]

for region in df_latest["region"].unique():
    df_region = df_latest[df_latest["region"] == region]
    fig = px.line(
        df_region,
        x="datetime",
        y="forecast_load_mw",
        color="forecast_date",
        title=f"Load Forecast (Latest Vintage) - {region}",
        labels={"forecast_load_mw": "Forecast Load (MW)", "datetime": "Date/Hour"},
    )
    fig.update_layout(template="plotly_dark", height=500)
    fig.show()

## Forecast Evolution (Vintages) by Region

In [6]:
# Show how forecasts for the most recent forecast_date evolved across execution dates
latest_forecast_date = df["forecast_date"].max()
df_evolution = df[df["forecast_date"] == latest_forecast_date]

for region in df_evolution["region"].unique():
    df_region = df_evolution[df_evolution["region"] == region]
    fig = px.line(
        df_region,
        x="hour_ending",
        y="forecast_load_mw",
        color="forecast_execution_date",
        title=f"Forecast Evolution for {latest_forecast_date} - {region}",
        labels={"forecast_load_mw": "Forecast Load (MW)", "hour_ending": "Hour Ending"},
    )
    fig.update_layout(template="plotly_dark", height=500)
    fig.show()

## All Regions Comparison (Latest Vintage)

In [7]:
df_latest_all = df.loc[df.groupby(["forecast_date", "hour_ending", "region"])["forecast_execution_datetime"].idxmax()]

fig = make_subplots(
    rows=len(df_latest_all["region"].unique()), cols=1,
    shared_xaxes=True,
    subplot_titles=sorted(df_latest_all["region"].unique()),
    vertical_spacing=0.06,
)

for i, region in enumerate(sorted(df_latest_all["region"].unique()), start=1):
    df_region = df_latest_all[df_latest_all["region"] == region]
    for fdate in sorted(df_region["forecast_date"].unique()):
        df_slice = df_region[df_region["forecast_date"] == fdate]
        fig.add_trace(
            go.Scatter(
                x=df_slice["datetime"],
                y=df_slice["forecast_load_mw"],
                name=str(fdate),
                legendgroup=str(fdate),
                showlegend=(i == 1),
            ),
            row=i, col=1,
        )
    fig.update_yaxes(title_text="MW", row=i, col=1)

fig.update_layout(
    title="Load Forecast - All Regions (Latest Vintage)",
    template="plotly_dark",
    height=350 * len(df_latest_all["region"].unique()),
)
fig.show()

## Daily Summary (OnPeak / OffPeak / Flat)

In [8]:
# OnPeak = HE 8-23, OffPeak = HE 1-7 & 24
df_latest_summary = df_latest.copy()
df_latest_summary["period"] = df_latest_summary["hour_ending"].apply(
    lambda h: "OnPeak" if 8 <= h <= 23 else "OffPeak"
)

summary = df_latest_summary.groupby(["forecast_date", "region", "period"])["forecast_load_mw"].mean().reset_index()
flat = df_latest_summary.groupby(["forecast_date", "region"])["forecast_load_mw"].mean().reset_index()
flat["period"] = "Flat"
summary = pd.concat([summary, flat], ignore_index=True)

fig = px.bar(
    summary,
    x="forecast_date",
    y="forecast_load_mw",
    color="period",
    facet_row="region",
    barmode="group",
    title="Avg Load Forecast - OnPeak / OffPeak / Flat",
    labels={"forecast_load_mw": "Avg MW", "forecast_date": "Forecast Date"},
)
fig.update_layout(template="plotly_dark", height=300 * len(summary["region"].unique()))
fig.show()